# Customer Churn Prediction - Complete 9-Phase ML Project
## Ready for Google Colab & Local Jupyter

This notebook provides a complete end-to-end machine learning pipeline covering all 9 project phases.

## Phase 1: Environment and Data Setup (9 Tasks)

In [ ]:
!pip install lightgbm xgboost joblib streamlit -q

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from lightgbm import LGBMClassifier

print('✓ All libraries installed and imported!')

In [ ]:
file_path = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
if not os.path.exists(file_path):
    try:
        from google.colab import files
        print('Upload your Telco Customer Churn CSV file:')
        uploaded = files.upload()
        file_path = list(uploaded.keys())[0]
    except Exception:
        pass

df = pd.read_csv(file_path)
print(f'✓ Dataset loaded successfully! Shape: {df.shape}')
df.head()

## Phase 2: Exploratory Data Analysis (9 Tasks)

In [ ]:
print('--- [Task 01] DATASET INFO ---')
df.info()

print('\n--- [Task 02] STATISTICAL SUMMARY ---')
display(df.describe())

print('\n--- [Task 03] MISSING VALUES ---')
print(df.isnull().sum())

In [ ]:
sns.set_style('whitegrid')
df_eda = df.copy()
df_eda['TotalCharges_num'] = pd.to_numeric(df_eda['TotalCharges'], errors='coerce')
df_eda['Churn_num'] = df_eda['Churn'].map({'Yes': 1, 'No': 0})

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
fig.suptitle('EDA: Target Distribution, Histograms, Box Plots & Correlation Heatmap', fontsize=16, fontweight='bold')

# Task 04: Count plot of target variable Churn
sns.countplot(data=df, x='Churn', palette=['#2ecc71', '#e74c3c'], ax=axes[0, 0])
axes[0, 0].set_title('Churn Distribution', fontweight='bold')

# Tasks 05 & 06: Histograms and Box Plots for numerical features
for i, col in enumerate(['tenure', 'MonthlyCharges', 'TotalCharges_num']):
    sns.histplot(df_eda[col].dropna(), kde=True, bins=30, ax=axes[0, i+1], color='#3498db')
    axes[0, i+1].set_title(f'Histogram: {col}', fontweight='bold')
    sns.boxplot(x=df_eda[col].dropna(), ax=axes[1, i+1], color='#9b59b6')
    axes[1, i+1].set_title(f'Box Plot: {col}', fontweight='bold')

# Task 09: Correlation Heatmap
corr = df_eda[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges_num', 'Churn_num']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', ax=axes[1, 0])
axes[1, 0].set_title('Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()

# Task 07: Count plots for categorical features
cat_cols_viz = ['Contract', 'PaymentMethod', 'InternetService', 'TechSupport']
fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for i, col in enumerate(cat_cols_viz):
    sns.countplot(data=df, x=col, hue='Churn', palette=['#2ecc71', '#e74c3c'], ax=axes[i])
    axes[i].set_title(f'Distribution: {col}', fontweight='bold')
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

# Task 08: Groupby Churn Rate
print('--- [Task 08] CHURN RATE BY CONTRACT TYPE ---')
print((df_eda.groupby('Contract')['Churn_num'].mean() * 100).round(2))

## Phase 3: Data Cleaning & Preprocessing (6 Tasks)

In [ ]:
print(f'Empty strings in TotalCharges: {(df["TotalCharges"] == " ").sum()}')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f'NaNs created in TotalCharges: {df["TotalCharges"].isnull().sum()}')

df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print(f'✓ Missing values remaining across all columns: {df.isnull().sum().sum()}')

## Phase 4: Feature Engineering (5 Tasks)

In [ ]:
# Task 01: tenure_group
df['tenure_group'] = pd.cut(
    df['tenure'], bins=[-1, 12, 48, 100], labels=['0-12 months', '13-48 months', '>48 months']
).astype(str)

# Task 02: monthly_charge_group
df['monthly_charge_group'] = pd.cut(
    df['MonthlyCharges'], bins=[0, 35, 70, 150], labels=['Low', 'Medium', 'High']
).astype(str)

# Task 03: has_dependents
df['has_dependents'] = (df['Dependents'] == 'Yes').astype(int)

# Task 04: total_services
service_cols = [
    'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df['total_services'] = (df[service_cols] == 'Yes').sum(axis=1)

# Task 05: Drop customerID
if 'customerID' in df.columns:
    df = df.drop('customerID', axis=1)

print('✓ Engineered features added successfully!')
df[['tenure', 'tenure_group', 'MonthlyCharges', 'monthly_charge_group', 'has_dependents', 'total_services']].head()

## Phase 5: Final Data Preparation (9 Tasks)

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn'].map({'Yes': 1, 'No': 0})

categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'total_services']

X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True, dtype=int)
feature_names = X_encoded.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test_scaled[numerical_cols] = scaler.transform(X_test[numerical_cols])

print(f'✓ Features encoded: {X_encoded.shape[1]} | Train shape: {X_train_scaled.shape} | Test shape: {X_test_scaled.shape}')

## Phase 6: Model Training & Comparison (10 Tasks)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'SVC': SVC(probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'LightGBM': LGBMClassifier(random_state=42, verbose=-1)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    preds = model.predict(X_test_scaled)
    probas = model.predict_proba(X_test_scaled)[:, 1]
    results[name] = {
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1 Score': f1_score(y_test, preds),
        'ROC-AUC': roc_auc_score(y_test, probas)
    }

comp_df = pd.DataFrame(results).T
display(comp_df.round(4))

comp_df.plot(kind='bar', figsize=(11, 5), ylim=(0.4, 1.0), colormap='viridis')
plt.title('Model Performance Comparison', fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Phase 7: Hyperparameter Tuning (6 Tasks)

In [ ]:
param_grid = {
    'n_estimators': [50, 100],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7]
}

grid_search = GridSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    param_grid=param_grid,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1
)
grid_search.fit(X_train_scaled, y_train)
print('✓ Best Parameters:', grid_search.best_params_)
final_model = grid_search.best_estimator_

## Phase 8: Final Evaluation & Interpretation (8 Tasks)

In [ ]:
y_pred_final = final_model.predict(X_test_scaled)
y_proba_final = final_model.predict_proba(X_test_scaled)[:, 1]
auc_final = roc_auc_score(y_test, y_proba_final)

print(classification_report(y_test, y_pred_final, target_names=['No Churn', 'Churn']))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
sns.heatmap(confusion_matrix(y_test, y_pred_final), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix', fontweight='bold')

fpr, tpr, _ = roc_curve(y_test, y_proba_final)
axes[1].plot(fpr, tpr, color='#e74c3c', lw=2, label=f'AUC = {auc_final:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()

importances = pd.Series(final_model.feature_importances_, index=feature_names).sort_values(ascending=False).head(10)
sns.barplot(x=importances.values, y=importances.index, palette='magma', ax=axes[2])
axes[2].set_title('Top 10 Feature Importances', fontweight='bold')
plt.tight_layout()
plt.show()

# Save artifacts for Phase 9 Streamlit App
joblib.dump(final_model, 'best_churn_model.joblib')
joblib.dump({
    'scaler': scaler,
    'feature_names': feature_names,
    'numerical_cols': numerical_cols,
    'categorical_cols': categorical_cols,
    'service_cols': service_cols
}, 'preprocessing_pipeline.joblib')
print('✓ Saved best_churn_model.joblib and preprocessing_pipeline.joblib')